In [ ]:
import yfinance as yf
import pandas as pd
import os

tickers = ['AAPL', 'MSFT', 'GOOG', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA']
start_date = '2020-01-01'
end_date = '2025-06-30'

os.makedirs('yfinance_data', exist_ok=True)

df = yf.download(tickers, start=start_date, end=end_date)
df.columns = ['{}_{}'.format(col[0], col[1]) for col in df.columns]  # flatten columns
df = df.reset_index()

df = (
    pd.melt(df, id_vars='Date', var_name='Price_Ticker', value_name='Value')
      .assign(Price_Type=lambda x: x.Price_Ticker.str.split('_').str[0],
              Ticker=lambda x: x.Price_Ticker.str.split('_').str[1])
      .drop(columns='Price_Ticker')
      .pivot_table(index=['Date', 'Ticker'], columns='Price_Type', values='Value')
      .reset_index()
)

df.to_csv('yfinance_data/combined_tickers.csv', index=False)
print("Saved combined data to yfinance_data/combined_tickers.csv")


In [ ]:
!pip install pandas_datareader

In [ ]:
import pandas_datareader.data as web
import datetime

# Define date range
start_date = datetime.datetime(2020, 1, 1)
end_date = datetime.datetime(2025, 6, 1)

# Extract data from FRED
cpi = web.DataReader('CPIAUCSL', 'fred', start_date, end_date)
fed_funds = web.DataReader('FEDFUNDS', 'fred', start_date, end_date)
unemployment = web.DataReader('UNRATE', 'fred', start_date, end_date)
gdp = web.DataReader('GDP', 'fred', start_date, end_date)

# Combine into a single DataFrame
econ_data = cpi.join([fed_funds, unemployment, gdp])
econ_data.columns = ['CPI', 'Federal_Funds_Rate', 'Unemployment_Rate', 'GDP']

# Show data
print(econ_data.head())

# Save to CSV
econ_data.to_csv('economic_data_fred.csv')


In [ ]:
import pandas as pd

# Load CSVs
stock_df = pd.read_csv('yfinance_data/combined_tickers.csv', parse_dates=['Date'])
econ_df = pd.read_csv('economic_data_fred.csv', parse_dates=['DATE'])

# Merge and immediately rename date to a unified name
merged_df = pd.merge(
    stock_df,
    econ_df,
    left_on='Date',
    right_on='DATE',
    how='left'
)

# Drop duplicate date column and rename
merged_df = merged_df.drop(columns=['DATE']).rename(columns={'Date': 'date'})

# Optional: add surrogate key
merged_df['stock_price_id'] = merged_df.index + 1

# Reorder columns using lowercase, standardized names
merged_df = merged_df.rename(columns={
    'Ticker': 'ticker',
    'Open': 'open',
    'High': 'high',
    'Low': 'low',
    'Close': 'close',
    'Volume': 'volume',
    'CPI': 'cpi_value',
    'Federal_Funds_Rate': 'interest_rate'
})

# Final column arrangement
fact_stock_prices = merged_df[
    ['stock_price_id', 'date', 'ticker', 'open', 'high', 'low', 'close', 'volume', 'cpi_value', 'interest_rate']
]

# Preview
print(fact_stock_prices.head())


In [ ]:
import pandas as pd

# Define the range of dates
date_range = pd.date_range(start='2020-01-01', end='2025-12-31')

# Create the dimension table
dim_date = pd.DataFrame({
    'date': date_range,
    'year': date_range.year,
    'month': date_range.month,
    'day': date_range.day,
    'weekday': date_range.day_name(),
    'quarter': date_range.quarter
})

# Show preview
print(dim_date.head())
